# Qwen3-ASR baseline (zero-shot)

Team Tokens · COE025 Project 3 · Finnish ASR

Pretrained Qwen3-ASR run in standard form, **no training by us**, so it is a **baseline**.
Scored with the shared `scripts/asr_eval.py` (same test clips, references, `normalize()`, WER/CER as every other system).

| | |
|---|---|
| Checkpoint | `Qwen/Qwen3-ASR-1.7B` (optional second run: `Qwen/Qwen3-ASR-0.6B`) |
| Finnish support | "Finnish" is in the `qwen-asr` package's `SUPPORTED_LANGUAGES` list (printed below) and on the model card language list |
| Language setting | forced `language="Finnish"` (no auto-detection) |
| Decoding | `qwen-asr` transformers backend, greedy, `max_new_tokens=256` |

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
`qwen-asr` pins its own `transformers` version, so run this notebook in a fresh runtime (not after the Whisper notebook).
At the end `qwen3_asr_results.zip` downloads: unzip it into the repo folder.
Optional: set `USE_DRIVE = True` to keep results on Google Drive, so that Run all continues after a disconnect.

In [ ]:
!pip -q install -U qwen-asr datasets jiwer soundfile

In [ ]:
# Get the repo (shared evaluation code + clip lists) and work inside it
import os, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"   # plain HTTP download; the Xet downloader can hang on Colab
if not os.path.isdir("/content/team-tokens-finnish-asr"):
    !git clone -q https://github.com/zehraoz36/team-tokens-finnish-asr /content/team-tokens-finnish-asr
else:
    !git -C /content/team-tokens-finnish-asr pull -q
%cd /content/team-tokens-finnish-asr
sys.path.insert(0, "scripts")
import asr_eval as E
print(E.hardware())
import torch
if not torch.cuda.is_available():
    raise SystemExit("NO GPU: Colab gave a CPU runtime (GPU quota used up?). "
                     "Runtime -> Change runtime type -> T4 GPU, or try later / another account.")

## Optional: save results to Google Drive

Off by default. A Colab disconnect (sleep, idle timeout) wipes the runtime, so with Drive off keep the
Mac awake and download the zip at the end. With `USE_DRIVE = True` (Drive access prompt), results are
kept on Drive and Run all after a disconnect skips the test sets that are already scored.

In [ ]:
# Keep results on Google Drive, so a disconnect does not lose finished test sets.
# Re-running the notebook then skips every test set that is already scored.
USE_DRIVE = False  # True: keep results on Google Drive (needs the Drive access prompt)
import shutil
DRIVE_DIR = "/content/drive/MyDrive/team-tokens-asr-results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        USE_DRIVE = False
        print("Drive not connected, continuing without it:", e)
for sub in (("scores", "predictions") if USE_DRIVE else ()):
    target = f"{DRIVE_DIR}/{sub}"
    os.makedirs(target, exist_ok=True)
    local = f"results/{sub}"
    if os.path.isdir(local) and not os.path.islink(local):
        for f in os.listdir(local):                      # repo files -> Drive (never overwrite)
            if not os.path.exists(f"{target}/{f}"):
                shutil.copy(f"{local}/{f}", target)
        shutil.rmtree(local)
    if not os.path.exists(local):
        os.symlink(target, local)
if USE_DRIVE:
    print("results ->", DRIVE_DIR, "| already there:", sorted(os.listdir(f"{DRIVE_DIR}/scores")))
else:
    print("results stay in this runtime only: download the zip at the end before closing")

In [ ]:
import torch, importlib.metadata as md
from qwen_asr import Qwen3ASRModel
from qwen_asr.inference.utils import SUPPORTED_LANGUAGES

print("qwen-asr", md.version("qwen-asr"), "| transformers", md.version("transformers"))
assert "Finnish" in SUPPORTED_LANGUAGES, SUPPORTED_LANGUAGES
print("Finnish supported ✓")

# bfloat16 only on GPUs that have it natively (compute capability >= 8: A100, L4).
# On a T4 (7.5) bfloat16 is only emulated and slow, so float16 is used there.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("dtype:", DTYPE)

In [ ]:
# Test sets: test_cv, test_parl, test_fleurs (16 kHz). References = `text` column.
test = E.load_test_sets()
for s, d in test.items():
    print(f"{s:12s} {len(d):5d} clips")

In [ ]:
def make_qwen(model_id, batch_size=16):
    model = Qwen3ASRModel.from_pretrained(model_id, dtype=DTYPE, device_map="cuda:0",
                                          max_inference_batch_size=batch_size, max_new_tokens=256)
    def transcribe(arrays):
        res = model.transcribe(audio=[(a, 16000) for a in arrays], language="Finnish")
        return [r.text for r in res]
    return model, transcribe

def settings(model_id):
    return (f"language=Finnish (forced), greedy, max_new_tokens=256, qwen-asr {md.version('qwen-asr')}, "
            f"transformers {md.version('transformers')}")

## Pipeline check on 8 **development** clips (not test)

Only checks that the output is Finnish text and not empty or garbled (e.g. float16 overflow).
No setting is tuned on the test sets.

In [ ]:
from datasets import load_dataset, Audio
dev = load_dataset(E.HF_DATASET, split="dev_cv").select(range(8)).cast_column("audio", Audio(sampling_rate=16000))

MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
model, fn = make_qwen(MODEL_ID)
hyps = fn([a["array"] for a in dev["audio"]])
for ref, hyp in zip(dev["text"], hyps):
    print("REF:", ref, "\nHYP:", E.normalize(hyp), "\n")
print("dev WER on 8 clips:", E.score(dev["text"], [E.normalize(h) for h in hyps])["wer"])

If the hypotheses above are empty or nonsense on a T4, set `DTYPE = torch.float32`, re-run the cell
above, and write that precision in the README.

## Test sets: Qwen3-ASR-1.7B

In [ ]:
SYSTEMS = []
NAME = "qwen3-asr-1.7b zero-shot"
E.evaluate_system(NAME, MODEL_ID, fn, test, role="baseline", settings=settings(MODEL_ID),
                  precision=str(DTYPE).split(".")[-1], batch_size=16)
SYSTEMS.append(NAME)
del model, fn; torch.cuda.empty_cache()

## Optional: Qwen3-ASR-0.6B

Set `RUN_06B = True` if there is time left (it is smaller and faster).

In [ ]:
RUN_06B = False
if RUN_06B:
    MODEL_ID = "Qwen/Qwen3-ASR-0.6B"
    NAME = "qwen3-asr-0.6b zero-shot"
    model, fn = make_qwen(MODEL_ID)
    E.evaluate_system(NAME, MODEL_ID, fn, test, role="baseline", settings=settings(MODEL_ID),
                      precision=str(DTYPE).split(".")[-1], batch_size=16)
    SYSTEMS.append(NAME)
    del model, fn; torch.cuda.empty_cache()

## Results and download

In [ ]:
display(E.show_results())
from google.colab import files
E.export_zip(SYSTEMS, "/content/qwen3_asr_results.zip")
files.download("/content/qwen3_asr_results.zip")